# Workers: speed-up as a function of the threshold

`bench_workers.zig` times `bufferToWorker` with `minBlocksForWorkers = 0`, so the workers always run, from 1,024 to 262,144 blocks per call.
At each size, the speed-up over one worker says whether the workers pay off there: the threshold `minBlocksForWorkers` should be the first size from which every worker count is above 1 (printed below, with the value of `main.zig`).
The dotted line marks where one call (8 + 1 bytes read and 64 bytes written per block) no longer fits in the 16 MB L3 cache.
Lines are medians, bands interquartile ranges over the 30 repetitions; the speed-up is paired by repetition.

In [ ]:
import re
import subprocess
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# the notebook sits in Claude Benchmark/<name>/, the repository two levels up
HERE = Path.cwd().resolve()
ROOT = HERE.parents[1]

MM = 1 / 25.4
INK, PAPER, GREY, SHADE = "#1f1d1a", "#faf8f3", "#77746e", "#d3cfc6"
BASELINE_GREY = "#a3a09a"
plt.rcParams.update({
    "font.family": "Liberation Sans", "font.size": 7, "axes.labelsize": 7,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.spines.top": False, "axes.spines.right": False,
})


def main_constant(name):
    """A constant of src/main.zig, such as `const MIN_BLOCKS_FOR_WORKERS = 16 * 1024;`, evaluated."""
    source = (ROOT / "src" / "main.zig").read_text(encoding="utf-8")
    expression = re.search(rf"^const {name} = ([0-9 *]+);", source, re.MULTILINE).group(1)
    return int(np.prod([int(factor) for factor in expression.split("*")]))


def run_benchmark(bench, csv):
    """Runs the benchmark (calls of bufferToWorker only) when its results are missing."""
    if not (HERE / csv).exists():
        subprocess.run(["zig", "run", "-OReleaseFast", "-mcpu=native", "--dep", "dequantizer",
                        f"-Mroot={HERE / bench}", f"-Mdequantizer={ROOT / 'src' / 'root.zig'}"],
                       cwd=ROOT, check=True)
    return pd.read_csv(HERE / csv)


In [ ]:
data = run_benchmark("bench_workers.zig", "workers.csv")
plate = int(data["blocks_per_plate"].iloc[0])
one = data[data.workers == 1].set_index(["blocks", "repetition"])["ns_per_call"]
data["speedup"] = one.loc[list(zip(data.blocks, data.repetition))].to_numpy() / data["ns_per_call"]
summary = data.groupby(["workers", "blocks"])["speedup"].quantile([0.25, 0.5, 0.75]).unstack()
sizes = sorted(data.blocks.unique())
counts = [count for count in sorted(data.workers.unique()) if count != 1]

# first size from which every worker count stays above one thread, at that size and every bigger one
above = {size: all(summary.loc[(count, bigger), 0.5] > 1 for count in counts for bigger in sizes if bigger >= size) for size in sizes}
crossover = next(size for size in sizes if above[size])
threshold = main_constant("MIN_BLOCKS_FOR_WORKERS")
print(f"crossover {crossover:,} blocks, main.zig uses {threshold:,}")
summary.round(2)

In [ ]:
magma = {2: 0.30, 4: 0.50, 6: 0.68, 12: 0.86}
L3_BYTES = 16 * 2**20  # Ryzen 5 7535U
fig, ax = plt.subplots(figsize=(140 * MM, 90 * MM))
fig.subplots_adjust(left=0.10, right=0.95, top=0.88, bottom=0.14)
for count in counts:
    colour = matplotlib.colormaps["magma"](magma[count])
    row = summary.loc[count]
    ax.fill_between(sizes, row[0.25], row[0.75], color=colour, alpha=0.13, lw=0)
    ax.plot(sizes, row[0.5], color=colour, lw=1.2, marker="o", ms=2.8, markeredgewidth=0, label=f"{count} workers")

# headroom so the labels at the top never sit on the curves
ax.set_ylim(0, summary[0.75].max() * 1.25)
ax.axhline(1, color=GREY, lw=0.8, ls=":")
ax.annotate("one thread", (sizes[0], 1.0), xytext=(0, 4), textcoords="offset points", color=GREY, fontsize=6.5, va="bottom")
# a block reads 8 + 1 bytes and writes 64 bytes of f32: from here one call no longer fits in the 16 MB L3 cache
l3_blocks = L3_BYTES / (8 + 1 + 64)
ax.axvline(l3_blocks, color=INK, lw=0.8, ls=":", zorder=0)
ax.annotate("L3 saturated", (l3_blocks, 1.0), xycoords=("data", "axes fraction"),
            xytext=(-6, -3), textcoords="offset points", color=INK, fontsize=6.5, va="top", ha="right")

ax.set_xscale("log", base=2)
ax.set_xticks(sizes[::2], [f"{size:,}" for size in sizes[::2]])
ax.set_xlabel("Blocks per call", labelpad=8)
ax.set_ylabel("Speed-up over one thread", labelpad=8)
fig.legend(loc="upper center", ncol=len(counts), frameon=False, bbox_to_anchor=(0.5, 0.985), columnspacing=2.2)
fig.savefig(HERE / "workers.png", dpi=300)